In [ ]:
from optics_design_workbench.jupyter_utils import *
import sympy as sy
from matplotlib.pyplot import *
from numpy import *

# Run tests with zero focal length

In [ ]:
results = []
with FreecadDocument() as f:
  for distribution in (
    'exp(-theta**2/0.01**2)',
    'exp(-theta**2/0.03**2)',
    '1',
    'cos(30*theta)**2',
    '2-abs(theta)',
  ):
    for thetaDomain in (
      '0, .1',
      '-.1, 0',
      '-.1, .1',
      '-.01, .02',
      '-.02, -.01',
      '.01, .02',
      '.01, .03',
    ):
      # calculate predicted power density
      thetas = linspace(-.1, .1, 500)
      powers = sy.lambdify('theta', distribution)(thetas)

      # update source parameters
      s = f.OpticalPointSource
      s.FocalLength = '0'
      s.PowerDensity = distribution
      s.ThetaDomain = thetaDomain
      s.PhiDomain = '0, 2*pi'
      s.RaysPerFan = 50
      s.Fans = 3

      # run fan simulation
      results.append([ s.PowerDensity.get(), 
                       s.ThetaDomain.get(),
                       f.runSimulation('fans'),
                       thetas, powers  ])

### Calc and plot diffs between simulation and expectation

In [ ]:
rmsErrs = []
for (dens, domain, r, thetas, powers) in results:
  figure(figsize=(7,2.5))
  if not hasattr(powers, '__len__'):
    powers = array([powers]*len(thetas))
  positions = 100*tan(thetas)

  # calc optimized scaling law between powers
  for fanI, powerFunc in r.loadHits('*').fanEstimatedPowerDensityFuncs().items():
    _positions, _powers = positions[powerFunc(positions)>1], powers[powerFunc(positions)>1]
    simulatedPowers = powerFunc(_positions)
    scaledRmsErr = lambda a: sqrt(mean(sorted((a*simulatedPowers - _powers)**2)[2:-2]))
    scaleA = scipy.optimize.minimize_scalar(scaledRmsErr).x

  for fanI, (_positions, _powers) in r.loadHits('*').fanEstimatedPowerDensities().items():
    _positions, _powers = _positions[1:-1], _powers[1:-1]

    _expectPowers = sy.lambdify('theta', dens)(atan(_positions/100))
    scaledRmsErr = lambda a: sqrt(mean(sorted((_expectPowers-a*_powers)**2)[1:-1]))
    scale = scipy.optimize.minimize_scalar(scaledRmsErr).x

    rmsErr = scaledRmsErr(scale)
    print(f'{rmsErr=}')
    rmsErrs.append(rmsErr)
    plot(_positions, scale*_powers, '.-', ms=1, lw=.2, label=f'fan #{fanI}, {rmsErr=:.0e}')

  #plot(_positions, _expectPowers, lw=5, alpha=.1)
  plot(positions, powers, lw=5, alpha=.2)
  xlim(-6, 6)
  grid(True)
  title(f'{dens}, {domain}')
  legend(loc=(1.01, 0))

In [ ]:
hist(log10(rmsErrs))

In [ ]:
median(rmsErrs), max(rmsErrs)

In [ ]:
assert median(rmsErrs) < 1e-2

In [ ]:
assert max(rmsErrs) < 0.1

# Run tests with infinite focal length

In [ ]:
results = []
with FreecadDocument() as f:
  for distribution in (
    'exp(-r**2/1**2)',
    'exp(-r**2/3**2)',
    '1',
    'cos(r/3)**2',
    '20-abs(r)',
  ):
    for radiusDomain in (
      '0, 10',
      '-10, 0',
      '-10, 10',
      '-1, 2',
      '-2, -1',
      '1.05, 2.123',
      '1.01, 3.321',
    ):
      # calculate predicted power density
      radii = linspace(-10, 10, 500)
      powers = sy.lambdify('r', distribution)(radii)

      # update source parameters
      s = f.OpticalPointSource
      s.PowerDensity = distribution
      s.FocalLength = 'inf'
      s.RadiusDomain = radiusDomain
      s.PhiDomain = '0, 2*pi'
      s.RaysPerFan = 70
      s.Fans = 3

      # run fan simulation
      results.append([ s.PowerDensity.get(), 
                       s.RadiusDomain.get(),
                       f.runSimulation('fans'),
                       radii, powers  ])

In [ ]:
rmsErrs = []
for (dens, domain, r, positions, powers) in results:
  figure(figsize=(7,2.5))
  if not hasattr(powers, '__len__'):
    powers = array([powers]*len(positions))

  # calc optimized scaling law between powers
  for fanI, powerFunc in r.loadHits('*').fanEstimatedPowerDensityFuncs().items():
    _positions, _powers = positions[powerFunc(positions)>1], powers[powerFunc(positions)>1]
    simulatedPowers = powerFunc(_positions)
    scaledRmsErr = lambda a: sqrt(mean(sorted((a*simulatedPowers - _powers)**2)[2:-2]))
    scaleA = scipy.optimize.minimize_scalar(scaledRmsErr).x

  for fanI, (_positions, _powers) in r.loadHits('*').fanEstimatedPowerDensities().items():
    _positions, _powers = _positions[1:-1], _powers[1:-1]

    _expectPowers = sy.lambdify('r', dens)(_positions)
    scaledRmsErr = lambda a: sqrt(mean(sorted((_expectPowers-a*_powers)**2)[1:-1]))
    scale = scipy.optimize.minimize_scalar(scaledRmsErr).x

    rmsErr = scaledRmsErr(scale)
    print(f'{rmsErr=}')
    rmsErrs.append(rmsErr)
    plot(_positions, scale*_powers, '.-', ms=1, lw=.2, label=f'fan #{fanI}, {rmsErr=:.0e}')

  #plot(_positions, _expectPowers, lw=5, alpha=.1)
  plot(positions, powers, lw=5, alpha=.2)
  xlim(-6, 6)
  grid(True)
  title(f'{dens}, {domain}')
  legend(loc=(1.01, 0))

In [ ]:
median(rmsErrs), max(rmsErrs)

In [ ]:
assert median(rmsErrs) < 1e-2

In [ ]:
assert max(rmsErrs) < 0.1